# Book Recommendation Engine using KNN

Completed solution for the freeCodeCamp **Machine Learning with Python** project.

Run the notebook from top to bottom. The final cell is the original freeCodeCamp test.


In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
# Make this cell safe to run more than once in Colab.
!rm -f book-crossings.zip BX-Books.csv BX-Book-Ratings.csv BX-Users.csv
!wget -q https://cdn.freecodecamp.org/project-data/books/book-crossings.zip -O book-crossings.zip
!unzip -oq book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding="ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'}
)

df_ratings = pd.read_csv(
    ratings_filename,
    encoding="ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'}
)

print("Books:", len(df_books))
print("Ratings:", len(df_ratings))

In [ ]:
# Remove users with fewer than 200 ratings and books with fewer than 100 ratings.
user_counts = df_ratings['user'].value_counts()
active_users = user_counts[user_counts >= 200].index

book_counts = df_ratings['isbn'].value_counts()
popular_books = book_counts[book_counts >= 100].index

filtered = df_ratings[
    df_ratings['user'].isin(active_users)
    & df_ratings['isbn'].isin(popular_books)
].copy()

# Attach book titles and ensure a user contributes at most one value per title.
filtered = filtered.merge(
    df_books[['isbn', 'title']],
    on='isbn',
    how='inner'
)
filtered = filtered.drop_duplicates(subset=['title', 'user'])

# Build the title x user matrix used by cosine-distance KNN.
book_user_matrix = filtered.pivot(
    index='title',
    columns='user',
    values='rating'
).fillna(0)

book_user_sparse = csr_matrix(book_user_matrix.values)

print("Active users:", len(active_users))
print("Popular ISBNs:", len(popular_books))
print("Model matrix:", book_user_matrix.shape)

In [ ]:
# Train the K-Nearest Neighbors model.
model_knn = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)
model_knn.fit(book_user_sparse)

In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book=""):
    if book not in book_user_matrix.index:
        return [book, []]

    query = book_user_matrix.loc[book].to_numpy().reshape(1, -1)

    n_neighbors = min(6, len(book_user_matrix))
    distances, indices = model_knn.kneighbors(
        query,
        n_neighbors=n_neighbors
    )

    recommendations = []

    for distance, index in zip(distances[0], indices[0]):
        title = book_user_matrix.index[index]

        # The first neighbor is the queried book itself (distance 0).
        if title == book:
            continue

        recommendations.append([title, float(distance)])

    # The challenge examples and test expect the five neighbors in reverse
    # of sklearn's ascending-distance order.
    recommendations = recommendations[:5][::-1]

    return [book, recommendations]

In [ ]:
# Optional example from the project statement.
print(
    get_recommends(
        "The Queen of the Damned (Vampire Chronicles (Paperback))"
    )
)

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()